# LLM-Based Action Planning for Network Intrusion Detection

This notebook uses saved prediction results, SHAP explanations, and RAG documents to generate LLM-based action plans for detected attacks.

In [1]:
# 1. Setup and Imports
# -----------------------------
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
from dotenv import load_dotenv

# Load environment variables from .env (e.g., OPENAI_API_KEY, OPENAI_MODEL, LLM_PROVIDER)
load_dotenv()

# RAG-related imports (install if needed: pip install sentence-transformers faiss-cpu PyPDF2)
try:
    from sentence_transformers import SentenceTransformer
    import faiss
    HAS_RAG_LIBS = True
except ImportError:
    print("WARNING: RAG libraries not installed. Install with: pip install sentence-transformers faiss-cpu PyPDF2")
    HAS_RAG_LIBS = False

PDF_LIB = None
try:
    import PyPDF2
    HAS_PDF_LIB = True
    PDF_LIB = "PyPDF2"
except ImportError:
    try:
        import pdfplumber
        HAS_PDF_LIB = True
        PDF_LIB = "pdfplumber"
    except ImportError:
        print("WARNING: PDF library not installed. Install with: pip install PyPDF2 or pip install pdfplumber")
        HAS_PDF_LIB = False

# Check required files (base logical names)
# Use only files under RAG_docs (no dependency on the top-level outputs/ folder).
rag_results_dir = Path("RAG_docs/results")
rag_knowledge_dir = Path("RAG_docs/knowledge")

required_files = {
    # Prediction / SHAP outputs (from model run), stored under RAG_docs/results
    "predictions": str(rag_results_dir / "sample_predictions_detailed_20260204_003608.json"),
    "shap_explanations": str(rag_results_dir / "sample_shap_explanations_20260204_003608.csv"),
    "metadata": str(rag_results_dir / "sample_predictions_metadata_20260204_003608.json"),
    # Base knowledge documents under RAG_docs/knowledge
    "rag_report_pdf": str(rag_knowledge_dir / "Targeted Cyber Intrusion Detection and Mitigation Strategies (Update B) _ CISA.pdf"),
    "rag_ddos_synthetic": str(rag_knowledge_dir / "ddos-synthetic-dataset.json"),
}

def _resolve_latest_rag_result(pattern: str) -> str:
    """Resolve to the newest matching file in RAG_docs/results for a given pattern."""
    if rag_results_dir.exists():
        matches = list(rag_results_dir.glob(pattern))
        if matches:
            latest = max(matches, key=lambda p: p.stat().st_mtime)
            print(f"  Resolved {pattern} -> {latest}")
            return str(latest)
    print(f"  WARNING: Could not resolve any files matching '{pattern}' in {rag_results_dir}")
    return ""

# Resolve versioned RAG result files commonly produced by the SHAP pipeline
required_files["predictions"] = _resolve_latest_rag_result("sample_predictions_detailed_*.json")
required_files["shap_explanations"] = _resolve_latest_rag_result("sample_shap_explanations_*.csv")
required_files["metadata"] = _resolve_latest_rag_result("sample_predictions_metadata_*.json")

print("Checking required files...")
for name, path in required_files.items():
    exists = os.path.exists(path)
    print(f"  {name}: {'✓' if exists else '✗'} {path}")
    if not exists:
        print(f"    WARNING: {path} not found!")

# Create output directory for action plans under RAG_docs/results
results_action_dir = rag_results_dir / "action_plans"
os.makedirs(results_action_dir, exist_ok=True)
print(f"\nOutput directory created: {results_action_dir}")
# -----------------------------

  Resolved sample_predictions_detailed_*.json -> RAG_docs\results\sample_predictions_detailed_20260204_003608.json
  Resolved sample_shap_explanations_*.csv -> RAG_docs\results\sample_shap_explanations_20260204_003608.csv
  Resolved sample_predictions_metadata_*.json -> RAG_docs\results\sample_predictions_metadata_20260204_003608.json
Checking required files...
  predictions: ✓ RAG_docs\results\sample_predictions_detailed_20260204_003608.json
  shap_explanations: ✓ RAG_docs\results\sample_shap_explanations_20260204_003608.csv
  metadata: ✓ RAG_docs\results\sample_predictions_metadata_20260204_003608.json
  rag_report_pdf: ✓ RAG_docs\knowledge\Targeted Cyber Intrusion Detection and Mitigation Strategies (Update B) _ CISA.pdf
  rag_ddos_synthetic: ✓ RAG_docs\knowledge\ddos-synthetic-dataset.json

Output directory created: RAG_docs\results\action_plans


In [2]:
# 2.5. Build RAG Knowledge Base from Rag_Docs, Outputs, and Summary
# -----------------------------

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file."""
    text_content = []
    
    if not HAS_PDF_LIB:
        return []
    
    try:
        if PDF_LIB == "pdfplumber":
            import pdfplumber
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        text_content.append(text)
        else:
            # Use PyPDF2
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                for page in pdf_reader.pages:
                    text = page.extract_text()
                    if text:
                        text_content.append(text)
    except Exception as e:
        print(f"  Warning: Could not extract text from {pdf_path}: {str(e)}")
    
    return text_content

def load_json_as_text(json_path):
    """Load JSON or JSONL file and convert to readable text format.

    Supports:
      - Single JSON object or array
      - JSON Lines (one JSON object per line)
    """
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            raw = f.read()

        # First, try standard JSON parse
        try:
            data = json.loads(raw)
            text_parts = []
            text_parts.append(f"Document: {os.path.basename(json_path)}")
            text_parts.append("=" * 80)

            if isinstance(data, dict):
                for key, value in data.items():
                    if isinstance(value, (dict, list)):
                        text_parts.append(f"\n{key}:")
                        text_parts.append(json.dumps(value, indent=2, ensure_ascii=False))
                    else:
                        text_parts.append(f"{key}: {value}")
            elif isinstance(data, list):
                text_parts.append(json.dumps(data, indent=2, ensure_ascii=False))
            else:
                text_parts.append(str(data))

            return "\n".join(text_parts)
        except json.JSONDecodeError:
            # Fallback: treat as JSON Lines (multiple JSON objects per line)
            records = []
            for line in raw.splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except Exception:
                    # If a line is not valid JSON, keep as raw text
                    records.append(line)

            text_parts = []
            text_parts.append(f"Document: {os.path.basename(json_path)} (JSONL)")
            text_parts.append("=" * 80)

            for idx, rec in enumerate(records, 1):
                text_parts.append(f"\nRecord {idx}:")
                if isinstance(rec, (dict, list)):
                    text_parts.append(json.dumps(rec, indent=2, ensure_ascii=False))
                else:
                    text_parts.append(str(rec))

            return "\n".join(text_parts)

    except Exception as e:
        print(f"  Warning: Could not load JSON from {json_path}: {str(e)}")
        return ""

def csv_to_text(csv_path):
    """Convert CSV to readable text format."""
    try:
        df = pd.read_csv(csv_path)
        text_parts = []
        text_parts.append(f"Document: {os.path.basename(csv_path)}")
        text_parts.append("=" * 80)
        
        # Add summary statistics
        text_parts.append(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")
        text_parts.append(f"\nColumns: {', '.join(df.columns.tolist())}")
        
        # Add data summary
        text_parts.append("\n\nData Summary:")
        text_parts.append(df.to_string(max_rows=50))
        
        # Add statistical summary for numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            text_parts.append("\n\nStatistical Summary:")
            text_parts.append(df[numeric_cols].describe().to_string())
        
        return "\n".join(text_parts)
    except Exception as e:
        print(f"  Warning: Could not load CSV from {csv_path}: {str(e)}")
        return ""

def chunk_text(text, chunk_size=1000, overlap=200):
    """Split text into overlapping chunks."""
    if len(text) <= chunk_size:
        return [text]
    
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    
    return chunks

# Build RAG knowledge base
print("=" * 80)
print("BUILDING RAG KNOWLEDGE BASE")
print("=" * 80)

rag_documents = []
rag_metadata = []

# 1. Load PDF files from RAG_docs/knowledge
print("\n1. Loading PDF files from RAG_docs/knowledge/...")
rag_docs_dir = rag_knowledge_dir
if rag_docs_dir.exists():
    pdf_files = list(rag_docs_dir.glob("*.pdf"))
    for pdf_file in pdf_files:
        print(f"  Processing: {pdf_file.name}")
        pages = extract_text_from_pdf(str(pdf_file))
        for idx, page_text in enumerate(pages):
            chunks = chunk_text(page_text)
            for chunk_idx, chunk in enumerate(chunks):
                rag_documents.append(chunk)
                rag_metadata.append({
                    "source": "RAG_docs",
                    "file": pdf_file.name,
                    "type": "PDF",
                    "page": idx + 1,
                    "chunk": chunk_idx + 1
                })
    print(f"  Loaded {len([m for m in rag_metadata if m['source'] == 'RAG_docs' and m['type'] == 'PDF'])} PDF chunks")
else:
    print("  Rag_Docs directory not found")

# 2. Load JSON files from RAG_docs/knowledge
print("\n2. Loading JSON files from RAG_docs/knowledge/...")
if rag_docs_dir.exists():
    json_files = list(rag_docs_dir.glob("*.json"))
    for json_file in json_files:
        print(f"  Processing: {json_file.name}")
        text_content = load_json_as_text(str(json_file))
        if text_content:
            chunks = chunk_text(text_content, chunk_size=2000)
            for chunk_idx, chunk in enumerate(chunks):
                rag_documents.append(chunk)
                rag_metadata.append({
                    "source": "RAG_docs",
                    "file": json_file.name,
                    "type": "JSON",
                    "chunk": chunk_idx + 1
                })
    print(f"  Loaded {len([m for m in rag_metadata if m['source'] == 'RAG_docs' and m['type'] == 'JSON'])} JSON chunks")
else:
    print("  Rag_Docs directory not found")

# 3–5. Skip loading outputs/ and summary/ into the RAG pile
print("\n3. Skipping outputs/ and summary/ for RAG; using only Rag_Docs as base knowledge.")

print(f"\n{'='*80}")
print(f"Total RAG documents loaded: {len(rag_documents)}")
print(f"{'='*80}")

# Create vector store if RAG libraries are available
rag_vector_store = None
rag_embeddings_model = None

if HAS_RAG_LIBS and len(rag_documents) > 0:
    print("\n6. Creating vector embeddings and FAISS index...")
    try:
        # Initialize embedding model
        print("  Loading sentence transformer model...")
        rag_embeddings_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Generate embeddings
        print(f"  Generating embeddings for {len(rag_documents)} documents...")
        embeddings = rag_embeddings_model.encode(rag_documents, show_progress_bar=True)
        
        # Create FAISS index
        print("  Creating FAISS index...")
        dimension = embeddings.shape[1]
        rag_vector_store = faiss.IndexFlatL2(dimension)
        rag_vector_store.add(embeddings.astype('float32'))
        
        print(f"  ✓ Vector store created with {rag_vector_store.ntotal} vectors")
        print(f"  ✓ Embedding dimension: {dimension}")
        
    except Exception as e:
        print(f"  ERROR creating vector store: {str(e)}")
        print("  RAG retrieval will use simple text matching instead")
        rag_vector_store = None
else:
    if not HAS_RAG_LIBS:
        print("\n  RAG libraries not available. Using simple text-based retrieval.")
    if len(rag_documents) == 0:
        print("\n  No documents loaded. RAG system will be empty.")

print("\nRAG knowledge base construction complete!")
# -----------------------------

BUILDING RAG KNOWLEDGE BASE

1. Loading PDF files from RAG_docs/knowledge/...
  Processing: Targeted Cyber Intrusion Detection and Mitigation Strategies (Update B) _ CISA.pdf
  Loaded 36 PDF chunks

2. Loading JSON files from RAG_docs/knowledge/...
  Processing: ddos-synthetic-dataset.json
  Loaded 192 JSON chunks

3. Skipping outputs/ and summary/ for RAG; using only Rag_Docs as base knowledge.

Total RAG documents loaded: 228

6. Creating vector embeddings and FAISS index...
  Loading sentence transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Generating embeddings for 228 documents...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  Creating FAISS index...
  ✓ Vector store created with 228 vectors
  ✓ Embedding dimension: 384

RAG knowledge base construction complete!


In [3]:
# 2.6. Save RAG Knowledge Base Summary
# -----------------------------

# Create summary of RAG knowledge base
rag_summary = {
    "total_documents": len(rag_documents),
    "sources": {},
    "file_types": {},
    "vector_store_available": rag_vector_store is not None,
    "embedding_model": "all-MiniLM-L6-v2" if rag_embeddings_model is not None else None
}

# Count by source
for meta in rag_metadata:
    source = meta.get("source", "unknown")
    file_type = meta.get("type", "unknown")
    
    if source not in rag_summary["sources"]:
        rag_summary["sources"][source] = 0
    rag_summary["sources"][source] += 1
    
    if file_type not in rag_summary["file_types"]:
        rag_summary["file_types"][file_type] = 0
    rag_summary["file_types"][file_type] += 1

# Save summary under RAG_docs/results/action_plans
summary_out_dir = results_action_dir
os.makedirs(summary_out_dir, exist_ok=True)
summary_path = summary_out_dir / "rag_knowledge_base_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(rag_summary, f, indent=2, ensure_ascii=False)

print("\n" + "="*80)
print("RAG KNOWLEDGE BASE SUMMARY")
print("="*80)
print(f"Total documents: {rag_summary['total_documents']}")
print(f"Vector store available: {rag_summary['vector_store_available']}")
print(f"\nDocuments by source:")
for source, count in rag_summary["sources"].items():
    print(f"  {source}: {count}")
print(f"\nDocuments by type:")
for file_type, count in rag_summary["file_types"].items():
    print(f"  {file_type}: {count}")
print(f"\nSummary saved to: {summary_path}")
print("="*80)
# -----------------------------


RAG KNOWLEDGE BASE SUMMARY
Total documents: 228
Vector store available: True

Documents by source:
  RAG_docs: 228

Documents by type:
  PDF: 36
  JSON: 192

Summary saved to: RAG_docs\results\action_plans\rag_knowledge_base_summary.json


In [4]:
# 2. Load Saved Results and RAG Documents
# -----------------------------

# Load prediction results
with open(required_files["predictions"], "r", encoding="utf-8") as f:
    predictions_data = json.load(f)

# Load SHAP explanations
shap_df = pd.read_csv(required_files["shap_explanations"])

# Load metadata
with open(required_files["metadata"], "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Load RAG documents (structured mitigation data now comes primarily from the RAG knowledge base)
rag_docs = {}

print(f"Loaded {len(predictions_data)} prediction results")
print(f"Loaded {len(shap_df)} SHAP explanations")
print(f"Loaded RAG documents: {list(rag_docs.keys())}")
print(f"\nParty names: {metadata.get('party_names', [])}")
print(f"Label mapping: {metadata.get('label_mapping', {})}")
# -----------------------------

Loaded 50 prediction results
Loaded 50 SHAP explanations
Loaded RAG documents: []

Party names: ['Volume_Rate_Sensor_Party1', 'Timing_Direction_Sensor_Party2', 'Timing_Direction_Sensor_Party3']
Label mapping: {'0': 'BENIGN', '1': 'BOT', '2': 'DDOS', '3': 'DOS', '4': 'FTPPATATOR', '5': 'OTHERS', '6': 'PORTSCAN', '7': 'SSHPATATOR', '8': 'WEBATTACK'}


In [5]:
# 3. RAG Knowledge Retrieval Functions
# -----------------------------

def search_rag_vector_store(query, top_k=5):
    """
    Search the RAG vector store for relevant documents using semantic similarity.
    """
    if rag_vector_store is None or rag_embeddings_model is None or len(rag_documents) == 0:
        return []
    
    try:
        # Encode query
        query_embedding = rag_embeddings_model.encode([query])
        
        # Search in FAISS index
        k = min(top_k, rag_vector_store.ntotal)
        distances, indices = rag_vector_store.search(query_embedding.astype('float32'), k)
        
        # Retrieve relevant documents with metadata
        results = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx < len(rag_documents):
                results.append({
                    "text": rag_documents[idx],
                    "metadata": rag_metadata[idx] if idx < len(rag_metadata) else {},
                    "distance": float(dist)
                })
        
        return results
    except Exception as e:
        print(f"  Warning: RAG vector search failed: {str(e)}")
        return []

def simple_text_search(query, top_k=5):
    """
    Fallback text-based search when vector store is not available.
    """
    if len(rag_documents) == 0:
        return []
    
    query_lower = query.lower()
    query_terms = query_lower.split()
    
    # Score documents by term frequency
    scored_docs = []
    for idx, doc in enumerate(rag_documents):
        doc_lower = doc.lower()
        score = sum(doc_lower.count(term) for term in query_terms)
        if score > 0:
            scored_docs.append({
                "text": doc,
                "metadata": rag_metadata[idx] if idx < len(rag_metadata) else {},
                "score": score
            })
    
    # Sort by score and return top_k
    scored_docs.sort(key=lambda x: x["score"], reverse=True)
    return scored_docs[:top_k]

def retrieve_mitigation_recommendations(attack_type, rag_docs):
    """
    Retrieve mitigation recommendations for a specific attack type from RAG documents.
    """
    recommendations = {
        "text_recommendations": "",
        "party_actions": [],
        "global_stats": [],
        "rag_retrieved_docs": []
    }
    
    # Extract from text file
    if "mitigation_summary" in rag_docs:
        content = rag_docs["mitigation_summary"]
        attack_upper = attack_type.upper()
        
        # Find section for this attack type
        start_marker = f"{attack_upper} Attack - Mitigation Recommendations"
        start_idx = content.find(start_marker)
        
        if start_idx != -1:
            # Find end of section (next attack or end of file)
            end_idx = content.find("\n\n", start_idx + 200)
            if end_idx == -1:
                end_idx = len(content)
            
            recommendations["text_recommendations"] = content[start_idx:end_idx]
    
    # Extract from agent mitigation CSV
    if "agent_mitigation_df" in rag_docs:
        df = rag_docs["agent_mitigation_df"]
        attack_rows = df[df['Class'].str.upper() == attack_type.upper()]
        
        if len(attack_rows) > 0:
            for _, row in attack_rows.iterrows():
                recommendations["party_actions"].append({
                    "party": row['Party'],
                    "domain": row['Domain'],
                    "contribution": row['Mean_contrib'],
                    "suggested_action": row['Suggested_Action'],
                    "is_dominant": row['Is_Dominant']
                })
    
    # Get global stats
    if "global_summary_df" in rag_docs:
        recommendations["global_stats"] = rag_docs["global_summary_df"].to_dict('records')
    
    # Search RAG vector store for additional context
    query = f"{attack_type} attack mitigation recommendations detection strategies"
    if rag_vector_store is not None:
        rag_results = search_rag_vector_store(query, top_k=5)
    else:
        rag_results = simple_text_search(query, top_k=5)
    
    # Format RAG results
    for result in rag_results:
        recommendations["rag_retrieved_docs"].append({
            "content": result["text"][:1000],  # Limit length
            "source": result["metadata"].get("source", "unknown"),
            "file": result["metadata"].get("file", "unknown"),
            "type": result["metadata"].get("type", "unknown")
        })
    
    return recommendations

def get_party_info(party_name, rag_docs):
    """
    Get detailed information about a specific party from RAG documents.
    """
    party_info = {
        "name": party_name,
        "domain": "",
        "global_contribution": 0.0
    }
    
    if "global_summary_df" in rag_docs:
        df = rag_docs["global_summary_df"]
        party_row = df[df['Party'] == party_name]
        if len(party_row) > 0:
            row = party_row.iloc[0]
            party_info["domain"] = row.get('Domain', '')
            party_info["global_contribution"] = row.get('Mean_contrib_All', 0.0)
    
    return party_info

print("RAG retrieval functions defined.")
# -----------------------------

RAG retrieval functions defined.


In [6]:
# 4. LLM Prompt Creation
# -----------------------------

def create_llm_prompt(prediction_result, shap_explanation, rag_knowledge, party_info):
    """
    Create a comprehensive prompt for LLM reasoning based on prediction, SHAP, and RAG.
    """
    attack_type = prediction_result.get('predicted_label', 'UNKNOWN')
    confidence = prediction_result.get('confidence', 0.0)
    
    # Format party contributions
    party_contribs = shap_explanation.get('party_contributions_pct', {})
    party_names_list = list(party_contribs.keys())
    party_values_list = list(party_contribs.values())
    
    prompt = f"""
You are a cybersecurity AI assistant analyzing a network intrusion detection result.

## DETECTION RESULTS
Attack Type Detected: {attack_type}
Confidence Score: {confidence:.2%}
Prediction Correct: {prediction_result.get('is_correct', False)}

All Class Probabilities:
{json.dumps(prediction_result.get('all_probabilities', {}), indent=2)}

## SHAP EXPLANATION (Which Sensor Detected This Attack)
Dominant Sensor/Party: {shap_explanation.get('dominant_party', 'Unknown')}
Dominant Contribution: {shap_explanation.get('dominant_contribution_pct', 0):.2%}

Party Contributions:
"""
    
    for party_name, contrib_pct in party_contribs.items():
        prompt += f"  - {party_name}: {contrib_pct:.2%}\n"
    
    prompt += f"""

## DOMINANT PARTY INFORMATION
Party Name: {party_info.get('name', 'Unknown')}
Domain: {party_info.get('domain', 'Unknown')}
Global Contribution: {party_info.get('global_contribution', 0):.2%}

## RETRIEVED KNOWLEDGE BASE (RAG)
### Mitigation Recommendations:
{rag_knowledge.get('text_recommendations', 'Not available')[:1000]}...

### Party-Specific Actions:
"""
    
    for party_action in rag_knowledge.get('party_actions', [])[:5]:  # Limit to 5 for prompt size
        prompt += f"""
Party: {party_action.get('party', 'Unknown')}
  Domain: {party_action.get('domain', 'Unknown')}
  Contribution: {party_action.get('contribution', 0):.2%}
  Suggested Action: {party_action.get('suggested_action', 'None')}
  Is Dominant: {party_action.get('is_dominant', False)}
"""
    
    # Add RAG-retrieved documents from vector store
    rag_docs = rag_knowledge.get('rag_retrieved_docs', [])
    if rag_docs:
        prompt += f"""

### Additional Context from RAG Knowledge Base (from PDFs, CSVs, and Summaries):
"""
        for idx, rag_doc in enumerate(rag_docs[:3], 1):  # Limit to top 3
            prompt += f"""
Document {idx} (Source: {rag_doc.get('source', 'unknown')}, File: {rag_doc.get('file', 'unknown')}, Type: {rag_doc.get('type', 'unknown')}):
{rag_doc.get('content', '')[:800]}...
"""
    
    prompt += """

## TASK
Based on the detection results, SHAP explanation, and retrieved knowledge base:

1. **Threat Assessment**: Analyze the threat level (Low/Medium/High/Critical)
   - Consider: confidence score, attack type severity, SHAP contribution strength

2. **Action Plan**: Recommend specific mitigation actions
   - Primary actions from the dominant sensor/party (based on RAG recommendations)
   - Supporting actions from other parties
   - Consider the retrieved knowledge base recommendations

3. **Reasoning**: Explain why these actions are appropriate
   - Why the dominant party detected this attack
   - How the actions address the specific attack type
   - Alignment with security best practices from RAG

4. **Framework Alignment**: Reference relevant security frameworks
   - NIST Cybersecurity Framework controls (if applicable)
   - OWASP Top 10 recommendations (if applicable)

5. **Execution Priority**: Determine action execution priority
   - Immediate (Critical threats, high confidence >0.9)
   - Fast-track (High threats, medium-high confidence 0.7-0.9)
   - Standard workflow (Medium threats, confidence 0.5-0.7)
   - Monitor only (Low threats or low confidence <0.5)

## OUTPUT FORMAT
Provide your response in the following JSON format:
{
  "threat_level": "Critical|High|Medium|Low",
  "threat_justification": "Brief explanation of threat level",
  "primary_actions": [
    "Action 1",
    "Action 2",
    ...
  ],
  "supporting_actions": [
    "Action 1",
    "Action 2",
    ...
  ],
  "reasoning": "Detailed explanation of why these actions are appropriate",
  "dominant_party_role": "Explanation of why the dominant party detected this attack",
  "framework_alignment": {
    "nist_controls": ["Control 1", "Control 2"],
    "owasp_recommendations": ["Recommendation 1", "Recommendation 2"]
  },
  "execution_priority": "Immediate|Fast-track|Standard|Monitor",
  "confidence_in_recommendations": "High|Medium|Low"
}

Begin your analysis:
"""
    
    return prompt

print("LLM prompt creation function defined.")
# -----------------------------

LLM prompt creation function defined.


In [7]:
# 5. LLM Integration (Real LLM API via .env)
# -----------------------------

def _parse_llm_json_response(raw_text: str) -> dict:
    """Best-effort extraction of a JSON object from the LLM response text."""
    raw_text = raw_text.strip()
    # Try direct parse first
    try:
        return json.loads(raw_text)
    except Exception:
        pass
    # Fallback: extract first {...} block
    try:
        start = raw_text.index('{')
        end = raw_text.rindex('}') + 1
        candidate = raw_text[start:end]
        return json.loads(candidate)
    except Exception:
        # Last resort: return a minimal structure
        return {
            "threat_level": "Unknown",
            "threat_justification": "Failed to parse structured JSON from LLM response.",
            "primary_actions": [raw_text[:500]],
            "supporting_actions": [],
            "reasoning": raw_text,
            "dominant_party_role": "Unknown",
            "framework_alignment": {"nist_controls": [], "owasp_recommendations": []},
            "execution_priority": "Standard",
            "confidence_in_recommendations": "Low",
        }


def call_llm_api(prompt: str) -> dict:
    """Call a real LLM API using credentials from .env and return a structured action plan.

    Expected .env variables:
      - LLM_PROVIDER (optional, default: "openai")
      - OPENAI_API_KEY (when provider == openai)
      - OPENAI_MODEL (optional, e.g. "gpt-4.1-mini" / "gpt-4o-mini")
      - OPENAI_TEMPERATURE (optional, e.g. 0.3)
    """
    provider = os.getenv("LLM_PROVIDER", "openai").lower()

    if provider == "openai":
        try:
            from openai import OpenAI
        except ImportError:
            raise ImportError("openai package not installed. Run: pip install openai")

        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise RuntimeError("OPENAI_API_KEY not set in environment or .env file.")

        model = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
        temperature = float(os.getenv("OPENAI_TEMPERATURE", "0.3"))

        # v1 client-based API per migration guide:
        # https://github.com/openai/openai-python/discussions/742
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a cybersecurity expert that returns strictly the requested JSON structure."},
                {"role": "user", "content": prompt},
            ],
            temperature=temperature,
        )

        # In v1, message is a ChatCompletionMessage object; use .content, not ["content"]
        llm_response_text = response.choices[0].message.content
        return _parse_llm_json_response(llm_response_text)

    # Fallback / other providers can be added here
    print("WARNING: Unsupported LLM_PROVIDER; falling back to simulated template response.")
    template_response = {
        "threat_level": "High",
        "threat_justification": "High confidence attack detection with significant SHAP contribution from dominant party",
        "primary_actions": [
            "Execute primary mitigation actions from dominant party based on RAG recommendations",
            "Monitor attack progression and collect additional evidence",
            "Log all related network flows for forensic analysis",
        ],
        "supporting_actions": [
            "Enable enhanced monitoring from other parties",
            "Collect additional evidence from non-dominant sensors",
            "Alert security team with detailed SHAP explanation",
        ],
        "reasoning": "Based on SHAP analysis, the dominant party has strong contribution to detection and RAG knowledge provides specific mitigation guidance.",
        "dominant_party_role": "The dominant party's sensor type is most effective for detecting this attack pattern based on its domain expertise and feature set.",
        "framework_alignment": {
            "nist_controls": ["PR.AC-5", "DE.AE-1", "RS.AN-1", "RS.CO-2"],
            "owasp_recommendations": ["Implement rate limiting", "Enable WAF rules", "Apply security patches"],
        },
        "execution_priority": "Fast-track",
        "confidence_in_recommendations": "Medium",
    }
    return template_response

print("LLM integration function defined (using real API via .env when configured).")
# -----------------------------

LLM integration function defined (using real API via .env when configured).


In [8]:
# 6. Process All Samples and Generate Action Plans
# -----------------------------

def process_sample_with_llm(sample_data, rag_docs, metadata):
    """
    Process a single sample through the LLM action planning pipeline.
    """
    # Get prediction result
    prediction_result = {
        "sample_id": sample_data.get("sample_id"),
        "predicted_label": sample_data.get("predicted_label"),
        "true_label": sample_data.get("true_label"),
        "confidence": sample_data.get("confidence"),
        "all_probabilities": sample_data.get("all_probabilities", {}),
        "is_correct": sample_data.get("is_correct", False)
    }
    
    # Get SHAP explanation
    shap_explanation = sample_data.get("shap_explanation", {})
    
    # Retrieve RAG knowledge
    attack_type = prediction_result.get("predicted_label", "UNKNOWN")
    rag_knowledge = retrieve_mitigation_recommendations(attack_type, rag_docs)
    
    # Get party info
    dominant_party = shap_explanation.get("dominant_party", "Unknown")
    party_info = get_party_info(dominant_party, rag_docs)
    
    # Create LLM prompt
    prompt = create_llm_prompt(prediction_result, shap_explanation, rag_knowledge, party_info)
    
    # Call LLM
    llm_response = call_llm_api(prompt)
    
    # Compile complete result
    result = {
        "sample_id": prediction_result["sample_id"],
        "prediction": prediction_result,
        "shap_explanation": shap_explanation,
        "rag_knowledge": {
            "attack_type": attack_type,
            "has_recommendations": len(rag_knowledge.get("text_recommendations", "")) > 0,
            "num_party_actions": len(rag_knowledge.get("party_actions", [])),
            "num_rag_retrieved_docs": len(rag_knowledge.get("rag_retrieved_docs", []))
        },
        "llm_action_plan": llm_response,
        "prompt_used": prompt,
        "generation_timestamp": datetime.now().isoformat()
    }
    
    return result

# Process all samples
print("="*80)
print("PROCESSING SAMPLES AND GENERATING ACTION PLANS")
print("="*80)

action_plans = []
num_samples = min(10, len(predictions_data))  # Process first 10 samples (adjust as needed)

print(f"\nProcessing {num_samples} samples...")
for idx, sample_data in enumerate(predictions_data[:num_samples]):
    print(f"\n[{idx+1}/{num_samples}] Processing sample {sample_data.get('sample_id')}...")
    
    try:
        result = process_sample_with_llm(sample_data, rag_docs, metadata)
        action_plans.append(result)
        
        # Print summary
        attack_type = result['prediction']['predicted_label']
        threat_level = result['llm_action_plan'].get('threat_level', 'Unknown')
        priority = result['llm_action_plan'].get('execution_priority', 'Unknown')
        print(f"  Attack: {attack_type} | Threat: {threat_level} | Priority: {priority}")
        
    except Exception as e:
        print(f"  ERROR processing sample {sample_data.get('sample_id')}: {str(e)}")
        continue

print(f"\n\nProcessed {len(action_plans)} samples successfully.")
# -----------------------------

PROCESSING SAMPLES AND GENERATING ACTION PLANS

Processing 10 samples...

[1/10] Processing sample 2379...
  Attack: BENIGN | Threat: Low | Priority: Monitor

[2/10] Processing sample 3599...
  Attack: BENIGN | Threat: Low | Priority: Monitor

[3/10] Processing sample 4178...
  Attack: PORTSCAN | Threat: High | Priority: Immediate

[4/10] Processing sample 8222...
  Attack: BENIGN | Threat: Low | Priority: Monitor

[5/10] Processing sample 8319...
  Attack: BENIGN | Threat: Low | Priority: Monitor

[6/10] Processing sample 8337...
  Attack: BENIGN | Threat: Low | Priority: Monitor

[7/10] Processing sample 11189...
  Attack: BENIGN | Threat: Low | Priority: Monitor

[8/10] Processing sample 11293...
  Attack: BENIGN | Threat: Low | Priority: Monitor

[9/10] Processing sample 12640...
  Attack: BENIGN | Threat: Low | Priority: Monitor

[10/10] Processing sample 15429...
  Attack: BENIGN | Threat: Low | Priority: Monitor


Processed 10 samples successfully.


In [9]:
# 7. Save Action Plans
# -----------------------------

# Save detailed action plans as JSON under RAG_docs/results/action_plans
output_file = results_action_dir / "llm_action_plans_detailed.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(action_plans, f, indent=2, ensure_ascii=False)

# Create summary DataFrame
summary_data = []
for plan in action_plans:
    summary_data.append({
        "sample_id": plan["sample_id"],
        "predicted_label": plan["prediction"]["predicted_label"],
        "true_label": plan["prediction"]["true_label"],
        "confidence": plan["prediction"]["confidence"],
        "is_correct": plan["prediction"]["is_correct"],
        "dominant_party": plan["shap_explanation"]["dominant_party"],
        "threat_level": plan["llm_action_plan"].get("threat_level", "Unknown"),
        "execution_priority": plan["llm_action_plan"].get("execution_priority", "Unknown"),
        "num_primary_actions": len(plan["llm_action_plan"].get("primary_actions", [])),
        "num_supporting_actions": len(plan["llm_action_plan"].get("supporting_actions", []))
    })

summary_df = pd.DataFrame(summary_data)
summary_csv_path = results_action_dir / "llm_action_plans_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

# Save action plans in readable format
readable_output = []
readable_output.append("="*80)
readable_output.append("LLM-BASED ACTION PLANS FOR NETWORK INTRUSION DETECTION")
readable_output.append("="*80)
readable_output.append(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
readable_output.append(f"Total samples: {len(action_plans)}")
readable_output.append("\n" + "="*80)

for plan in action_plans:
    readable_output.append(f"\n\n{'='*80}")
    readable_output.append(f"SAMPLE {plan['sample_id']}")
    readable_output.append(f"{'='*80}")
    
    pred = plan["prediction"]
    shap = plan["shap_explanation"]
    llm = plan["llm_action_plan"]
    
    readable_output.append(f"\nDetection:")
    readable_output.append(f"  Attack Type: {pred['predicted_label']}")
    readable_output.append(f"  Confidence: {pred['confidence']:.2%}")
    readable_output.append(f"  True Label: {pred['true_label']}")
    readable_output.append(f"  Correct: {pred['is_correct']}")
    
    readable_output.append(f"\nSHAP Explanation:")
    readable_output.append(f"  Dominant Party: {shap['dominant_party']}")
    readable_output.append(f"  Dominant Contribution: {shap['dominant_contribution_pct']:.2%}")
    
    readable_output.append(f"\nLLM Action Plan:")
    readable_output.append(f"  Threat Level: {llm.get('threat_level', 'Unknown')}")
    readable_output.append(f"  Execution Priority: {llm.get('execution_priority', 'Unknown')}")
    readable_output.append(f"\n  Primary Actions:")
    for i, action in enumerate(llm.get('primary_actions', []), 1):
        readable_output.append(f"    {i}. {action}")
    readable_output.append(f"\n  Supporting Actions:")
    for i, action in enumerate(llm.get('supporting_actions', []), 1):
        readable_output.append(f"    {i}. {action}")
    readable_output.append(f"\n  Reasoning:")
    readable_output.append(f"    {llm.get('reasoning', 'N/A')}")
    readable_output.append(f"\n  Framework Alignment:")
    readable_output.append(f"    NIST: {', '.join(llm.get('framework_alignment', {}).get('nist_controls', []))}")
    readable_output.append(f"    OWASP: {', '.join(llm.get('framework_alignment', {}).get('owasp_recommendations', []))}")

readable_content = "\n".join(readable_output)
readable_txt_path = results_action_dir / "llm_action_plans_readable.txt"
with open(readable_txt_path, "w", encoding="utf-8") as f:
    f.write(readable_content)

print("="*80)
print("ACTION PLANS SAVED")
print("="*80)
print(f"\nFiles saved:")
print(f"  - {output_file}")
print(f"  - {summary_csv_path}")
print(f"  - {readable_txt_path}")
print(f"\nTotal action plans generated: {len(action_plans)}")

# Robust summary printing (handle empty or partial DataFrame)
if not summary_df.empty and 'threat_level' in summary_df.columns:
    print(f"\nThreat level distribution:")
    print(summary_df['threat_level'].value_counts())
else:
    print("\nThreat level distribution: no data (no action plans or column missing)")

if not summary_df.empty and 'execution_priority' in summary_df.columns:
    print(f"\nExecution priority distribution:")
    print(summary_df['execution_priority'].value_counts())
else:
    print("\nExecution priority distribution: no data (no action plans or column missing)")
# -----------------------------

ACTION PLANS SAVED

Files saved:
  - RAG_docs\results\action_plans\llm_action_plans_detailed.json
  - RAG_docs\results\action_plans\llm_action_plans_summary.csv
  - RAG_docs\results\action_plans\llm_action_plans_readable.txt

Total action plans generated: 10

Threat level distribution:
threat_level
Low     9
High    1
Name: count, dtype: int64

Execution priority distribution:
execution_priority
Monitor      9
Immediate    1
Name: count, dtype: int64
